# Ultra-Scale Playbook 训练系统 · 第 11/14 课

> 状态：**未开始**  
> 一次只完成一课；未通过前不要打开下一课答案。

## 统一完成标准

代码 4 分、Q1～Q3 各 2 分，通过线 8/10。必须解释正确性边界、显存/通信公式中的单位与分片维度；未实际运行的内容只能标记为静态审查。

# 第 11 课：5D 并行

- 对应官方章节：5D Parallelism in a Nutshell（含 Scope and focus、Summarizing it all）
- 前置：第 4～10 课全部
- 状态：未开始

## 本课目标

完成后你需要能够：

- 列出五个并行维度及其切分的对象与维度：DP(batch)、TP(hidden)、SP/CP(sequence)、PP(layers)、EP(experts)，外加 ZeRO-1/2/3。
- 说出各技术的显存节省对象与主要瓶颈（教材总表）。
- 按教材的三步流程为一个给定的模型 + 集群设计并行配置。
- 解释组合规则：TP 节点内、PP/ZeRO-3 跨节点、ZeRO-1/2 与 PP 兼容、ZeRO-3 与 PP 很少组合。

## 核心概念

### 1. 五维总览（教材总表）

| 方法 | 显存节省对象 | 并行/切分维度 | 主要缺点 |
|---|---|---|---|
| DP | 激活（减小本地 batch） | batch | 受最大 batch 限制 |
| PP | 模型参数 | 模型层 | 空闲 bubble、调度复杂 |
| TP+SP | 参数与激活 | 隐藏维/序列维 | 需要高带宽通信 |
| CP | 激活 | 序列维 | attention 通信开销 |
| EP | 专家参数 | 专家维 | 需 MoE、路由通信 |
| ZeRO-1 | 优化器状态 | DP 副本间分片 | 参数通信开销 |
| ZeRO-2 | 优化器状态 + 梯度 | DP 副本间分片 | 参数通信开销 |
| ZeRO-3 | 优化器状态 + 梯度 + 参数 | DP 副本间分片 | 参数通信开销 |

要点：ZeRO 系列**只沿 DP 维**分片（激活无法分片）；TP/SP/CP/EP 是模型结构相关的分片；PP 与 ZeRO-3 是"模型无关"的权重分片（但一个传激活、一个传权重）。

### 2. 组合规则（面试核心）

- **TP（+SP）只放节点内**（TP ≤ 8，NVLink）：它的通信在计算关键路径上，跨节点带宽不够。
- **PP 与 ZeRO-3 适合跨节点**：PP 每层边界只传少量激活；ZeRO-3 的通信可以 prefetch 隐藏。
- **ZeRO-1/2 与 PP 天然互补**：DeepSeek-V3 = PP + ZeRO-1。ZeRO-3 与 PP 很少组合——通信都沿模型深度走，两者叠加需要极大 gbs 摊薄；若组合，ZeRO-3 应在 PP 微批次期间保持权重驻留内存，减少重复 all-gather。
- **CP 管长序列、EP 管 MoE**，与上述维度正交。
- 规模经验：512+ 卡纯 DP/ZeRO-3 效率下降 → 加 TP 或 PP；1024+ 卡推荐 TP=8 + ZeRO-2 + PP。

### 3. 教材的配置三步流程（第 12 课细化）

1. **装进内存**：<10B 模型单技术即可；10–100B 用 TP=8+PP 或 TP=8+ZeRO-3 或纯 ZeRO-3；超长序列加 CP；MoE 加 EP；GPU 少就开全量重计算 + 梯度累积。
2. **达到目标 gbs**：`gbs_tokens = mbs × grad_acc × dp × seq`；batch 不够就加 dp/grad_acc（长序列可加 CP），batch 太大就减 dp。
3. **优化吞吐**：TP 升到节点大小、ZeRO-3 加 dp、通信瓶颈转 PP、调 mbs。

### 4. 典型配置锚点

- Llama 3（405B）：TP=8 × PP 多节点 × DP/ZeRO × 可选 EP。
- DeepSeek-V3：PP + ZeRO-1 + EP（256 专家）+ CP 可选。
- 记住"先让模型装下（TP/PP/ZeRO-3），再凑 batch（dp/grad_acc/CP），最后调吞吐"。

## 代码填空题

为一个模型 + 集群设计配置：给定约束，找出满足"装得下 + 达到 gbs"的 (tp, pp, cp, dp, zero, grad_acc)。


In [ ]:
def validate_config(
    cfg: dict,                  # {"tp","pp","cp","ep","zero","mbs","grad_acc"}
    num_params: int,
    gpu_memory: int,            # bytes
    num_nodes: int,
    gpus_per_node: int,
    num_heads: int,
    target_gbs_tokens: int,
    seq_len: int,
    num_layers: int,            # 用于粗估激活（每层 ~2·seq·mbs·h，h 需另行传入或简化）
    hidden: int,
) -> tuple[bool, list[str]]:
    """检查配置是否合法。返回 (是否可行, 违规原因列表)。"""
    problems = []
    num_gpus = num_nodes * gpus_per_node

    # 1) 并行度乘积必须等于（或不超过）GPU 总数
    used = cfg["dp"] * cfg["tp"] * cfg["pp"] * cfg["cp"] * cfg["ep"]
    if used != num_gpus:
        problems.append(f"并行度乘积 {used} != GPU 总数 {num_gpus}")

    # 2) TP 必须在节点内，且不超过注意力头数
    if cfg["tp"] > gpus_per_node:
        problems.append("TP 超过单节点 GPU 数")
    if cfg["tp"] > num_heads:
        problems.append("TP 超过注意力头数")

    # 3) gbs（token）达标：mbs * grad_acc * dp * seq
    gbs = cfg["mbs"] * cfg["grad_acc"] * cfg["dp"] * seq_len
    if gbs < target_gbs_tokens:
        problems.append(f"gbs={gbs} < 目标 {target_gbs_tokens}")

    # 4) 每卡静态显存（BF16+Adam，无 FP32 梯度累积）
    # 注意：参数/梯度/优化器状态先被 TP 分片（/tp），ZeRO 项再沿 DP 分片（/dp）。
    #   zero=0：全部 /tp；zero=1：参数和梯度 /tp，优化器状态 /(dp·tp)；
    #   zero=2：参数 /tp，梯度和优化器状态 /(dp·tp)；zero=3：全部 /(dp·tp)
    k, psi, tp = 12, num_params, cfg["tp"]
    if cfg["zero"] == 0:
        static = (2 * psi + 2 * psi + k * psi) / tp
    elif cfg["zero"] == 1:
        static = ______                     # 填空
    elif cfg["zero"] == 2:
        static = ______                     # 填空
    else:
        static = ______                     # 填空

    # 5) 激活粗估（每层两处 s·mbs·h，BF16 2 字节；忽略 attention 二次项）
    act_per_gpu = (2 * num_layers * seq_len // cfg["cp"] * cfg["mbs"] * hidden * 2)
    total = static + act_per_gpu + 2 * 1024**3    # 预留 2 GiB 运行时/CUDA context
    if total > gpu_memory:
        problems.append(f"估算显存 {total/1e9:.1f} GB > 单卡 {gpu_memory/1e9:.0f} GB")

    return (not problems), problems


def search_configs(
    num_params: int, gpu_memory: int, num_nodes: int, gpus_per_node: int,
    num_heads: int, num_layers: int, hidden: int,
    target_gbs_tokens: int, seq_len: int, mbs: int,
    zero_stages=(0, 1, 2, 3), tp_choices=(1, 2, 4, 8), cp_choices=(1, 2, 4, 8),
    pp_choices=(1, 2, 4, 8),
):
    """在 TP/PP/CP/ZeRO 网格上搜索可行配置（dp 由总数反推，ep=1）。"""
    num_gpus = num_nodes * gpus_per_node
    found = []
    for tp in tp_choices:
        for pp in pp_choices:
            for cp in cp_choices:
                ep = 1
                if num_gpus % (tp * pp * cp * ep) != 0:
                    continue
                dp = num_gpus // (tp * pp * cp * ep)
                for zero in zero_stages:
                    # grad_acc 取满足 gbs 的最小整数
                    grad_acc = max(1, -(-(target_gbs_tokens) // (mbs * dp * seq_len)))
                    cfg = dict(tp=tp, pp=pp, cp=cp, ep=ep, zero=zero,
                               mbs=mbs, grad_acc=grad_acc, dp=dp)
                    ok, problems = validate_config(cfg, num_params, gpu_memory,
                                                   num_nodes, gpus_per_node,
                                                   num_heads, target_gbs_tokens,
                                                   seq_len, num_layers, hidden)
                    if ok:
                        found.append((cfg, problems))
    return found


if __name__ == "__main__":
    # 案例：70B 模型（h=8192, L=80, 头 64），16 节点 × 8×80GB H100，
    # 目标 gbs = 4M tokens、seq = 4096、mbs = 1
    found = search_configs(
        num_params=70_000_000_000, gpu_memory=80 * 1024**3,
        num_nodes=16, gpus_per_node=8, num_heads=64, num_layers=80,
        hidden=8192, target_gbs_tokens=4_000_000, seq_len=4096, mbs=1,
    )
    print(f"找到 {len(found)} 个可行配置；前 5 个：")
    for cfg, _ in found[:5]:
        print(f"  dp={cfg['dp']:3d} tp={cfg['tp']} pp={cfg['pp']:2d} "
              f"cp={cfg['cp']} zero={cfg['zero']} grad_acc={cfg['grad_acc']} "
              f"(并行度乘积 {cfg['dp']*cfg['tp']*cfg['pp']*cfg['cp']})")


## 三个问答题


### Q1

为什么 TP 组必须留在节点内，而 PP/ZeRO-3 可以跨节点？请分别从"通信内容/频率"与"能否重叠"两个角度回答。CP 与 EP 在跨节点问题上各处于什么位置？


### Q2

用教材三步流程为以下场景设计配置：70B 稠密模型、16 节点 × 8×H100（80 GB）、seq=4096、目标 gbs=4M tokens、mbs=1。写出你选择的 (dp, tp, pp, cp, zero, grad_acc) 与理由；至少给出一个"装得下但吞吐差"的反例配置并说明为什么差。


### Q3

为什么 ZeRO-3 与 PP 很少组合，而 ZeRO-1/2 与 PP 互补（DeepSeek-V3 即 PP+ZeRO-1）？如果坚持组合 ZeRO-3+PP，ZeRO-3 应该做哪一项关键配置来减少通信浪费？

## 检查与通过标准

总分 10 分：代码正确 4 分（约束检查完整、gbs 公式、搜索能产出合理配置）、三题各 2 分、通过线 8 分。

一票否决项：

- 五个维度的切分对象/维度搞混（如把 CP 说成切 batch）。
- 认为 TP 可以跨节点扩展而没有明显代价。
- gbs 公式漏乘 dp 或 seq。
- 说不出 ZeRO-3 与 PP 为何很少组合。


## 官方主参考

- [Ultra-Scale Playbook](https://huggingface.co/spaces/nanotron/ultrascale-playbook)
- [PyTorch distributed documentation](https://pytorch.org/docs/stable/distributed.html)